# imports

In [1]:
import mysql.connector
import pandas as pd
import numpy as np

from pathlib import Path


In [2]:
# Path relative to Scripts/
data_dir = Path("../Data")
input_file = data_dir / "2026-07-06_2_bank_dataset_cleaned.csv"

df = pd.read_csv(input_file, encoding="utf-8-sig", index_col=0)

print(f"Dataset loaded from: {input_file.resolve()}")
print(f"Shape: {df.shape}")

Dataset loaded from: /Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/04_Simulador/ProjecteData/Equip_32/Data/2026-07-06_2_bank_dataset_cleaned.csv
Shape: (10975, 19)


In [3]:
df = df.reset_index()  # converteix l'índex en columna normal
df.rename(columns={'index': 'id'}, inplace=True)  # per si el nom no és 'id'

In [4]:
df

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,no_previous_contact,had_previous_contact
0,1,59.0,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,1,1,0
1,2,56.0,admin.,married,secondary,no,45,no,no,unknown,5,may,1381,1,-1,0,unknown,1,1,0
2,3,41.0,technician,married,secondary,no,1270,yes,no,unknown,5,may,1381,1,-1,0,unknown,1,1,0
3,4,55.0,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,1,1,0
4,5,54.0,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10970,10983,40.0,management,married,secondary,no,8486,no,no,unknown,6,may,260,3,-1,0,unknown,0,1,0
10971,10984,53.0,management,married,tertiary,no,20772,no,no,cellular,4,feb,715,1,-1,0,unknown,0,1,0
10972,10985,55.0,blue-collar,married,primary,no,3297,yes,yes,telephone,30,apr,96,1,-1,0,unknown,0,1,0
10973,10986,41.0,management,married,tertiary,no,9,yes,no,cellular,22,jul,82,3,-1,0,unknown,0,1,0


# Data Transformations

# 1 Demografical clustering

In [5]:
# Age_group
bins   = [17, 25, 35, 50, 65, 100]
labels = [
    "Young (18-25)",
    "Young Adult (26-35)",
    "Adult (36-50)",
    "Middle-Aged (51-65)",
    "Senior (65+)"
]

df["age_group"] = pd.cut(
    df["age"],
    bins=bins,
    labels=labels,
    right=True    # right-closed intervals: (17,25] includes 25
)

In [6]:
counts = df["age_group"].value_counts().sort_index()
print(counts)
print(f"\nUnassigned (NaN): {df['age_group'].isna().sum()}")

age_group
Young (18-25)           444
Young Adult (26-35)    3840
Adult (36-50)          4255
Middle-Aged (51-65)    2039
Senior (65+)            397
Name: count, dtype: int64

Unassigned (NaN): 0


In [7]:
age_summary = (
    df.groupby("age_group", observed=True)["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

age_summary["n_clients"] = df.groupby("age_group", observed=True).size()
print(age_summary)

deposit              pct_no  pct_yes  n_clients
age_group                                      
Young (18-25)        0.2815   0.7185        444
Young Adult (26-35)  0.5138   0.4862       3840
Adult (36-50)        0.5803   0.4197       4255
Middle-Aged (51-65)  0.5135   0.4865       2039
Senior (65+)         0.1965   0.8035        397


In [8]:
# Financial Burden

# "unknown" is treated as 0 (no burden assumed)
# This is a deliberate modelling choice — document it in the notebook

burden_map = {"yes": 1, "no": 0, "unknown": 0}

df["housing_score"]  = df["housing"].map(burden_map)
df["loan_score"]     = df["loan"].map(burden_map)

In [9]:
df["financial_burden"] = (
    df["housing_score"] +
    df["loan_score"]
)

In [10]:
burden_labels = {
    0: "No burden",
    1: "Low burden",
    2: "High burden",
    
}

df["financial_burden_label"] = df["financial_burden"].map(burden_labels)

In [11]:
burden_summary = (
    df.groupby("financial_burden_label")["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

burden_summary["n_clients"] = df.groupby("financial_burden_label").size()

# Sort by score for readability
burden_summary = burden_summary.reindex(burden_labels.values())
print(burden_summary)

deposit                 pct_no  pct_yes  n_clients
financial_burden_label                            
No burden               0.3961   0.6039       5185
Low burden              0.6205   0.3795       4972
High burden             0.6760   0.3240        818


In [12]:
# education

print(df["education"].value_counts())
print(f"\nUnknown count: {(df['education'] == 'unknown').sum()}")


education
secondary    5385
tertiary     3637
primary      1466
unknown       487
Name: count, dtype: int64

Unknown count: 487


In [13]:
# Ordinal scale: unknown → NaN (excluded from ranking)
# primary=1, secondary=2, tertiary=3

education_order = {
    "primary"   : 1,
    "secondary" : 2,
    "tertiary"  : 3,
    "unknown"   : None
}

df["education_rank"] = df["education"].map(education_order)

In [14]:
education_labels = {
    "primary"   : "Primary",
    "secondary" : "Secondary",
    "tertiary"  : "Tertiary",
    "unknown"   : "Unknown"
}

df["education_label"] = df["education"].map(education_labels)

In [15]:
edu_summary = (
    df.groupby("education_label")["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

edu_summary["n_clients"] = df.groupby("education_label").size()

# Sort by ordinal rank
order = ["Primary", "Secondary", "Tertiary", "Unknown"]
edu_summary = edu_summary.reindex(order)
print(edu_summary)

deposit          pct_no  pct_yes  n_clients
education_label                            
Primary          0.5969   0.4031       1466
Secondary        0.5456   0.4544       5385
Tertiary         0.4520   0.5480       3637
Unknown          0.4825   0.5175        487


In [16]:
#jobs

print(df["job"].value_counts())
print(f"\nUnknown count: {(df['job'] == 'unknown').sum()}")

job
management       2531
blue-collar      1904
technician       1791
admin.           1317
services          905
retired           769
self-employed     398
student           358
unemployed        346
entrepreneur      319
housemaid         267
unknown            70
Name: count, dtype: int64

Unknown count: 70


In [17]:
job_profile_map = {
    "admin."       : "White Collar",
    "management"   : "White Collar",
    "technician"   : "White Collar",
    "blue-collar"  : "Blue Collar",
    "housemaid"    : "Blue Collar",
    "services"     : "Blue Collar",
    "entrepreneur" : "Self-Employed",
    "self-employed": "Self-Employed",
    "retired"      : "Retired",
    "student"      : "Student",
    "unemployed"   : "Unemployed",
    "unknown"      : "Unknown"
}

df["job_profile"] = df["job"].map(job_profile_map)

In [18]:
job_summary = (
    df.groupby("job_profile")["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

job_summary["n_clients"] = df.groupby("job_profile").size()

order = ["White Collar", "Blue Collar", "Self-Employed", "Unemployed", "Retired", "Student", "Unknown"]
job_summary = job_summary.reindex(order)
print(job_summary)

deposit        pct_no  pct_yes  n_clients
job_profile                              
White Collar   0.5091   0.4909       5639
Blue Collar    0.6144   0.3856       3076
Self-Employed  0.5676   0.4324        717
Unemployed     0.4162   0.5838        346
Retired        0.3303   0.6697        769
Student        0.2514   0.7486        358
Unknown        0.5143   0.4857         70


In [19]:
month_map = {
    'jan': 'January', 'feb': 'February', 'mar': 'March',
    'apr': 'April', 'may': 'May', 'jun': 'June',
    'jul': 'July', 'aug': 'August', 'sep': 'September',
    'oct': 'October', 'nov': 'November', 'dec': 'December'
}
df['month'] = df['month'].map(month_map)

In [20]:
df['week_of_month'] = ((df['day'] - 1) // 7) + 1
# Resultat: 1, 2, 3, 4, 5

In [21]:
bins = [0, 3, 10, 20, 30, float('inf')]
labels = ['0-3', '4-10', '11-20', '21-30', '+30']
df['campaign_group'] = pd.cut(df['campaign'], bins=bins, labels=labels)

In [22]:
# Opció 2: Reset de l'índex com a columna neta
df = df.reset_index(drop=True)
df.index = df.index + 1  # comença des de 1 en lloc de 0


# CSV export

In [23]:
# Drop auto-generated index column if it was imported as a column
if "index" in df.columns:
    df = df.drop(columns=["index"])

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
 

# Drop intermediate scoring columns
cols_to_drop = [
    "housing_score", 
    "loan_score"
]

df_export = df.drop(columns=cols_to_drop)

# Export transformed dataset
output_dir = Path("../Data")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "2026-07-06_3_bank_dataset_transformed.csv"

# Export
df_export.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV saved to: {output_file.resolve()}")
print(f"Shape: {df_export.shape}")
print(f"Columns: {df_export.columns.tolist()}")

CSV saved to: /Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/04_Simulador/ProjecteData/Equip_32/Data/2026-07-06_3_bank_dataset_transformed.csv
Shape: (10975, 28)
Columns: ['id', 'age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'deposit', 'no_previous_contact', 'had_previous_contact', 'age_group', 'financial_burden', 'financial_burden_label', 'education_rank', 'education_label', 'job_profile', 'week_of_month', 'campaign_group']
